# 3. Metin Ön İşleme
Bu notebook, NLP modelleri için ekstra özellik çıkarımı ve metin temizleme adımlarını içerir.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

tqdm.pandas()

print("Veri yükleniyor...")
df = pd.read_csv('data/reviews_cleaned.csv')
print(f"Veri yüklendi, satır sayısı: {len(df)}")


## 3.1 Ek Feature Üretimi

In [ ]:
print("Feature'lar çıkarılıyor...")
df['text'] = df['text'].astype(str)

df['review_length'] = df['text'].str.len()
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
df['exclamation_count'] = df['text'].str.count('!')
df['question_count'] = df['text'].str.count('\\?')

def avg_word_len(text):
    words = text.split()
    if len(words) == 0: return 0
    return sum(len(word) for word in words) / len(words)

df['avg_word_length'] = df['text'].progress_apply(avg_word_len)

def upper_ratio(text):
    if len(text) == 0: return 0
    uppers = sum(1 for c in text if c.isupper())
    return uppers / len(text)

df['uppercase_ratio'] = df['text'].progress_apply(upper_ratio)

def get_sentiment(text):
    blob = TextBlob(text)
    return pd.Series([blob.sentiment.polarity, blob.sentiment.subjectivity])

print("TextBlob sentiment hesaplanıyor...")
df[['sentiment_polarity', 'sentiment_subjectivity']] = df['text'].progress_apply(get_sentiment)

print("\nSınıflara göre feature ortalamaları:")
features = ['review_length', 'word_count', 'exclamation_count', 'question_count', 
            'avg_word_length', 'uppercase_ratio', 'sentiment_polarity', 'sentiment_subjectivity']
print(df.groupby('label')[features].mean())


## 3.2 Metin Temizleme Fonksiyonu

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    # 1. Küçük harfe çevir
    text = text.lower()
    
    # 2. HTML tag ve URL temizle
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 3. Emoji ve unicode kaldır
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # 4. Noktalama ve özel karakter kaldır
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 5. Sayıları kaldır
    text = re.sub(r'\d+', '', text)
    
    # 6. Fazla boşlukları temizle
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 7. Tokenize et
    tokens = word_tokenize(text)
    
    # 8-9-10. Stopword kaldır, Lemmatization, 2 karakterden kısa olanları at
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) >= 2]
    
    # 11. Birleştir ve döndür
    return " ".join(cleaned_tokens)


## 3.3 Önce/Sonra Karşılaştırması

In [ ]:
sample_reviews = df['text'].sample(5, random_state=42).tolist()
cleaned_samples = [clean_text(text) for text in sample_reviews]

comparison_df = pd.DataFrame({
    'Orijinal Metin': sample_reviews,
    'Temizlenmiş Metin': cleaned_samples
})

pd.set_option('display.max_colwidth', None)
display(comparison_df)


## 3.4 Tüm Veriye Uygulama

In [ ]:
print("Tüm metinler temizleniyor...")
df['cleaned_text'] = df['text'].progress_apply(clean_text)

initial_len = len(df)
df = df[df['cleaned_text'].str.strip() != '']
df = df.dropna(subset=['cleaned_text'])

dropped = initial_len - len(df)
print(f"\nBoş kaldığı için atılan satır sayısı: {dropped}")
print(f"Kalan veri boyutu: {len(df)}")


## 3.5 Feature İstatistikleri

In [ ]:
print("Feature istatistikleri (Mean ve Std):")
stats = df.groupby('label')[features].agg(['mean', 'std']).round(3)
display(stats)

plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x='label', y='sentiment_polarity', palette='Set2')
plt.title('Sınıflara Göre Sentiment Polarity (Duygu Skoru)')
plt.xlabel('Sınıf (0=Kötü, 1=Orta, 2=İyi)')
plt.ylabel('Polarity')
plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/preprocessing_features.png')
plt.show()


## 3.6 Kaydet

In [ ]:
output_csv = 'data/reviews_preprocessed.csv'
df.to_csv(output_csv, index=False)

os.makedirs('models', exist_ok=True)
joblib.dump(clean_text, 'models/preprocessor.pkl')

print(f"\n[OK] Kaydedildi: {output_csv}")
print("[OK] Kaydedildi: models/preprocessor.pkl")

display(df.head())
